## 02 — Train classifier (scenario_id)

This notebook trains a baseline text classifier:
- input: evidence-derived prompt
- target: `scenario_id`

It uses a configurable Hugging Face `Trainer` pipeline.


In [14]:
from pathlib import Path
import sys
import pandas as pd

THIS_NOTEBOOK_DIR = Path.cwd()
SRC_DIR = (THIS_NOTEBOOK_DIR.parent / "src").resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from capstone_training.config import ClassificationConfig, SplitConfig
from capstone_training.train_classifier import train_classifier_from_dataframe


In [17]:
# debugging cell [ignore]
import importlib
import capstone_training.train_classifier as tc
importlib.reload(tc)
from capstone_training.train_classifier import train_classifier_from_dataframe

In [18]:
DATASETS_DIR = (THIS_NOTEBOOK_DIR.parent / "artifacts" / "datasets").resolve()
df = pd.read_parquet(DATASETS_DIR / "classification.parquet")

cfg = ClassificationConfig(
    model_name="distilbert-base-uncased",
    output_dir=THIS_NOTEBOOK_DIR.parent / "artifacts" / "classifier_distilbert",
    num_train_epochs=2,
    train_batch_size=8,
    eval_batch_size=8,
)

result = train_classifier_from_dataframe(df, cfg=cfg, split=SplitConfig(test_size=0.2, val_size=0.1, seed=42))
print("Saved to:", result["output_dir"])
print("Test accuracy:", result["metrics_test"].accuracy)


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 17620.90it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/Users/najelalarcon/Desktop/Multi-Agent-Collaboration-System/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argumen

Epoch,Training Loss,Validation Loss,Accuracy,Precision Macro,Recall Macro,F1 Macro,Top3 Accuracy
1,0.004787,0.002842,1.000000,1.000000,1.000000,1.000000,1.000000
2,0.002212,0.001498,1.000000,1.000000,1.000000,1.000000,1.000000


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.89it/s]
/Users/najelalarcon/Desktop/Multi-Agent-Collaboration-System/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.94it/s]
There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].
/Users/najelalarcon/Desktop/Multi-Agent-Collaboration-System/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.44it/s]


Saved to: /Users/najelalarcon/Desktop/Multi-Agent-Collaboration-System/najel-data/training/artifacts/classifier_distilbert
Test accuracy: 1.0
